# Urban Mission Planning Challenge

**Pipeline:** Satellite image → Road segmentation (U-Net) → Cost map → Dijkstra pathfinding → Submission

| Module | Description |
|--------|-------------|
| 0 | Setup & Data Loading |
| 1 | Road Segmentation (U-Net, ResNet34 encoder) |
| 2 | Pathfinding (Dijkstra on cost map) |
| 3 | Validation & Submission |

**Scoring:** `1000 - PathLength - 50 × Violations` — off-road segments cost 50× more than path length.

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install -q segmentation-models-pytorch albumentations tifffile

In [ ]:
import os, sys, json, math, time, platform
import numpy as np
from pathlib import Path

import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
from skimage.graph import route_through_array
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Module 0 — Setup & Data Loading

In [ ]:
# --- Configuration ---
DATA_ROOT = Path("ump_data")
REF_SATS  = DATA_ROOT / "reference" / "sats"
REF_MAPS  = DATA_ROOT / "reference" / "maps"
REF_SOLS  = DATA_ROOT / "reference" / "solutions"
TEST_SATS = DATA_ROOT / "test" / "sats"

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0  # safe for Windows & Colab

# Training hyper-parameters
PATCH_SIZE        = 512
BATCH_SIZE        = 8
PATCHES_PER_IMAGE = 8
NUM_EPOCHS        = 30
LR                = 1e-4
WEIGHT_DECAY      = 1e-4

print(f"Device: {DEVICE}")
print(f"Data root exists: {DATA_ROOT.exists()}")

In [ ]:
# --- Data loading helpers ---

def load_image(path):
    """Load satellite TIFF -> (H, W, 3) uint8 numpy array."""
    img = tifffile.imread(str(path))
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    return img.astype(np.uint8)

def load_mask(path):
    """Load road mask TIFF -> (H, W) boolean numpy array."""
    img = tifffile.imread(str(path))
    if img.ndim == 3:
        img = img[:, :, 0]
    return img > 127

def load_metadata(path):
    with open(path) as f:
        return json.load(f)

def load_solution(path):
    """Load solution JSON -> list of [x, y] coords."""
    with open(path) as f:
        data = json.load(f)
    if isinstance(data, list):
        return data
    return data.get("path") or data.get("points")

In [ ]:
# --- Load metadata & sanity check ---
ref_meta  = load_metadata(DATA_ROOT / "reference_metadata.json")
test_meta = load_metadata(DATA_ROOT / "test_metadata.json")

print(f"Training samples: {len(ref_meta)}")
print(f"Test samples:     {len(test_meta)}")
print(f"\nFirst entry: {json.dumps(ref_meta[0], indent=2)}")

# CRITICAL: verify coordinate convention  [x, y] in JSON  vs  [row, col] in numpy
mask_001 = load_mask(REF_MAPS / "train_001_map.tiff")
sx, sy = ref_meta[0]["start"]          # JSON [x, y]
print(f"\ntrain_001 start (x={sx}, y={sy})")
print(f"mask[y={sy}, x={sx}] = {mask_001[sy, sx]}")   # numpy [row, col]
assert mask_001[sy, sx], "Start point NOT on road — coordinate bug!"
print("Coordinate sanity check PASSED")

In [ ]:
# --- Visualize a training sample ---
img_001 = load_image(REF_SATS / "train_001.tiff")
sol_001 = load_solution(REF_SOLS / "train_001_solution.json")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_001);           axes[0].set_title("Satellite Image")
axes[1].imshow(mask_001, cmap="gray"); axes[1].set_title("Road Mask")

axes[2].imshow(img_001)
px = [p[0] for p in sol_001]; py = [p[1] for p in sol_001]
axes[2].plot(px, py, "r-", linewidth=1, alpha=0.7)
axes[2].plot(sx, sy, "go", markersize=10, label="Start")
gx, gy = ref_meta[0]["goal"]
axes[2].plot(gx, gy, "ro", markersize=10, label="Goal")
axes[2].legend(); axes[2].set_title("Image + Reference Path")
plt.tight_layout(); plt.show()

print(f"Image shape: {img_001.shape}  dtype: {img_001.dtype}")
print(f"Mask  shape: {mask_001.shape}  road%: {100*mask_001.mean():.1f}%")
print(f"Solution: {len(sol_001)} points, "
      f"length: {sum(math.hypot(a[0]-b[0],a[1]-b[1]) for a,b in zip(sol_001,sol_001[1:])):.0f} px")

## Module 1 — Road Segmentation (U-Net)

In [ ]:
class RoadDataset(Dataset):
    """Extracts random patches from satellite / mask pairs.  Caches images in RAM."""

    def __init__(self, indices, patch_size=PATCH_SIZE,
                 patches_per_image=PATCHES_PER_IMAGE, transform=None):
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.transform = transform
        self.indices = indices
        self._cache = {}

    def _load(self, list_idx):
        if list_idx not in self._cache:
            i = self.indices[list_idx]
            img_id = f"train_{i:03d}"
            image = load_image(REF_SATS / f"{img_id}.tiff")
            mask  = load_mask(REF_MAPS  / f"{img_id}_map.tiff")
            self._cache[list_idx] = (image, mask)
        return self._cache[list_idx]

    def __len__(self):
        return len(self.indices) * self.patches_per_image

    def __getitem__(self, idx):
        image, mask = self._load(idx // self.patches_per_image)
        h, w = image.shape[:2]
        ps = self.patch_size
        cy = np.random.randint(0, h - ps + 1)
        cx = np.random.randint(0, w - ps + 1)

        img_p = image[cy:cy+ps, cx:cx+ps].copy()
        msk_p = mask[cy:cy+ps, cx:cx+ps].astype(np.float32).copy()

        if self.transform:
            t = self.transform(image=img_p, mask=msk_p)
            img_p, msk_p = t["image"], t["mask"]

        if isinstance(msk_p, torch.Tensor):
            msk_p = msk_p.unsqueeze(0)
        else:
            msk_p = torch.from_numpy(msk_p).float().unsqueeze(0)

        return img_p, msk_p

In [ ]:
# --- Augmentations & data loaders ---
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

train_indices = list(range(1, 41))   # train_001 – train_040
val_indices   = list(range(41, 51))  # train_041 – train_050

train_dataset = RoadDataset(train_indices, transform=train_transform)
val_dataset   = RoadDataset(val_indices, patches_per_image=4, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Train: {len(train_dataset)} patches  ({len(train_indices)} imgs x {PATCHES_PER_IMAGE})")
print(f"Val:   {len(val_dataset)} patches  ({len(val_indices)} imgs x 4)")

In [ ]:
# --- Model, loss, optimizer ---
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(DEVICE)

class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
    def forward(self, pred, target):
        return 0.5 * self.bce(pred, target) + 0.5 * self.dice(pred, target)

criterion = CombinedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# --- Training loop ---
best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "val_iou": []}

for epoch in range(NUM_EPOCHS):
    t0 = time.time()

    # ---- train ----
    model.train()
    running = 0.0
    for imgs, msks in train_loader:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), msks)
        loss.backward()
        optimizer.step()
        running += loss.item()
    train_loss = running / len(train_loader)

    # ---- validate ----
    model.eval()
    vloss_sum, iou_sum, n = 0.0, 0.0, 0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            out = model(imgs)
            vloss_sum += criterion(out, msks).item()
            preds = (torch.sigmoid(out) > 0.5).float()
            inter = (preds * msks).sum(dim=(1, 2, 3))
            union = preds.sum(dim=(1, 2, 3)) + msks.sum(dim=(1, 2, 3)) - inter
            iou_sum += (inter / (union + 1e-8)).sum().item()
            n += imgs.size(0)
    val_loss = vloss_sum / len(val_loader)
    val_iou  = iou_sum / n
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)

    dt = time.time() - t0
    tag = ""
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), "best_road_model.pth")
        tag = "  *saved*"
    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS}  train={train_loss:.4f}  "
          f"val={val_loss:.4f}  IoU={val_iou:.4f}  {dt:.1f}s{tag}")

print(f"\nBest val IoU: {best_val_iou:.4f}")

In [ ]:
# --- Training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["train_loss"], label="Train")
ax1.plot(history["val_loss"],   label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")
ax2.plot(history["val_iou"])
ax2.set_xlabel("Epoch"); ax2.set_ylabel("IoU"); ax2.set_title("Validation IoU")
plt.tight_layout(); plt.show()

In [ ]:
# --- Load best model & sliding-window inference ---
model.load_state_dict(torch.load("best_road_model.pth", map_location=DEVICE, weights_only=True))
model.eval()
print("Loaded best model weights")

def predict_mask(model, image, patch_size=PATCH_SIZE, overlap=128, threshold=0.5):
    """Sliding-window inference with overlap averaging."""
    model.eval()
    h, w = image.shape[:2]
    stride = patch_size - overlap

    pred_sum = np.zeros((h, w), dtype=np.float32)
    count    = np.zeros((h, w), dtype=np.float32)

    normalize = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # patch positions (include right / bottom edges)
    ys = sorted(set(list(range(0, h - patch_size + 1, stride)) + [h - patch_size]))
    xs = sorted(set(list(range(0, w - patch_size + 1, stride)) + [w - patch_size]))

    with torch.no_grad():
        for y in ys:
            for x in xs:
                patch  = image[y:y+patch_size, x:x+patch_size]
                normed = normalize(image=patch)["image"]
                tensor = torch.from_numpy(normed).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
                out    = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
                pred_sum[y:y+patch_size, x:x+patch_size] += out
                count[y:y+patch_size, x:x+patch_size]    += 1.0

    return (pred_sum / np.maximum(count, 1)) >= threshold

def postprocess_mask(mask):
    """Morphological cleanup: close gaps, remove noise, thin noise."""
    mask = binary_closing(mask, disk(3))
    mask = remove_small_objects(mask, min_size=100)
    mask = binary_opening(mask, disk(2))
    return mask

In [ ]:
# --- Quick inference test on train_001 ---
print("Inference on train_001 ...")
t0 = time.time()
pred_001 = postprocess_mask(predict_mask(model, img_001))
print(f"Time: {time.time()-t0:.1f}s")

iou = (pred_001 & mask_001).sum() / ((pred_001 | mask_001).sum() + 1e-8)
print(f"Full-image IoU vs GT: {iou:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_001);                  axes[0].set_title("Satellite")
axes[1].imshow(mask_001, cmap="gray");    axes[1].set_title("GT Mask")
axes[2].imshow(pred_001, cmap="gray");    axes[2].set_title(f"Predicted (IoU={iou:.3f})")
plt.tight_layout(); plt.show()

## Module 2 — Pathfinding (Dijkstra on cost map)

In [ ]:
# --- Pathfinding helpers ---

def build_cost_map(mask):
    """Road pixels are cheap (cheaper near centerlines); off-road is 1000."""
    dt   = distance_transform_edt(mask)
    cost = np.full(mask.shape, 1000.0, dtype=np.float64)
    road = mask > 0
    cost[road] = 1.0 / (1.0 + dt[road])
    return cost

def snap_to_road(x, y, mask, radius=50):
    """Find nearest road pixel to (x, y) within radius."""
    if mask[y, x]:
        return x, y
    h, w = mask.shape
    for r in range(1, radius + 1):
        y0, y1 = max(0, y - r), min(h, y + r + 1)
        x0, x1 = max(0, x - r), min(w, x + r + 1)
        region = mask[y0:y1, x0:x1]
        if region.any():
            ry, rx = np.where(region)
            ry += y0; rx += x0
            best = np.argmin((rx - x)**2 + (ry - y)**2)
            return int(rx[best]), int(ry[best])
    return x, y

def find_path(cost_map, start_xy, goal_xy, mask):
    """Dijkstra shortest path. Returns list of [x, y] coords."""
    sx, sy = start_xy
    gx, gy = goal_xy

    # snap to road if needed
    sx_s, sy_s = snap_to_road(sx, sy, mask)
    gx_s, gy_s = snap_to_road(gx, gy, mask)

    # route_through_array uses (row, col) = (y, x)
    indices, weight = route_through_array(
        cost_map, start=(sy_s, sx_s), end=(gy_s, gx_s), fully_connected=True
    )

    # convert (row, col) -> [x, y]
    path = [[int(c), int(r)] for r, c in indices]

    # prepend / append if we snapped
    if (sx, sy) != (sx_s, sy_s):
        path.insert(0, [sx, sy])
    if (gx, gy) != (gx_s, gy_s):
        path.append([gx, gy])

    return path

In [ ]:
# --- Evaluation helpers (matching evaluate.py logic) ---

def compute_path_length(path):
    return sum(math.hypot(a[0]-b[0], a[1]-b[1]) for a, b in zip(path, path[1:]))

def count_violations(path, mask):
    """Count off-road point & segment violations (same logic as evaluate.py)."""
    h, w = mask.shape
    off_pts, off_segs = 0, 0

    for x, y in path:
        if x < 0 or x >= w or y < 0 or y >= h or not mask[y, x]:
            off_pts += 1

    for a, b in zip(path, path[1:]):
        dx, dy = b[0] - a[0], b[1] - a[1]
        steps = max(abs(dx), abs(dy), 1)
        bad = False
        for i in range(steps + 1):
            t = i / steps
            px = int(round(a[0] + dx * t))
            py = int(round(a[1] + dy * t))
            if px < 0 or px >= w or py < 0 or py >= h or not mask[py, px]:
                bad = True
                break
        if bad:
            off_segs += 1

    return off_pts, off_segs

In [ ]:
# --- Test pathfinding on GT masks (isolates pathfinding from segmentation) ---
print("Pathfinding on ground-truth masks\n")
print(f"{'ID':<12} {'GT Len':>8} {'Pred Len':>9} {'Diff':>7} {'ViolPt':>7} {'ViolSeg':>8} {'Time':>6}")
print("-" * 62)

for i in range(5):
    m = ref_meta[i]
    gt_mask  = load_mask(REF_MAPS / f"{m['id']}_map.tiff")
    gt_sol   = load_solution(REF_SOLS / f"{m['id']}_solution.json")
    cost_map = build_cost_map(gt_mask)

    t0 = time.time()
    pred_path = find_path(cost_map, m["start"], m["goal"], gt_mask)
    dt = time.time() - t0

    gt_len   = compute_path_length(gt_sol)
    pred_len = compute_path_length(pred_path)
    vp, vs   = count_violations(pred_path, gt_mask)

    print(f"{m['id']:<12} {gt_len:>8.0f} {pred_len:>9.0f} {pred_len-gt_len:>+7.0f} "
          f"{vp:>7} {vs:>8} {dt:>5.1f}s")

In [ ]:
# --- Visualize path comparison (train_001, GT mask) ---
m = ref_meta[0]
gt_mask  = load_mask(REF_MAPS / f"{m['id']}_map.tiff")
gt_sol   = load_solution(REF_SOLS / f"{m['id']}_solution.json")
cost_map = build_cost_map(gt_mask)
our_path = find_path(cost_map, m["start"], m["goal"], gt_mask)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(load_image(REF_SATS / f"{m['id']}.tiff"))
ax.plot([p[0] for p in gt_sol],   [p[1] for p in gt_sol],   "g-", lw=2, alpha=.7, label="GT")
ax.plot([p[0] for p in our_path], [p[1] for p in our_path], "r--", lw=2, alpha=.7, label="Ours")
ax.plot(m["start"][0], m["start"][1], "wo", ms=12, mec="k")
ax.plot(m["goal"][0],  m["goal"][1],  "w*", ms=15, mec="k")
ax.legend(fontsize=12); ax.set_title(f"{m['id']}: GT vs Our path (GT mask)")
plt.tight_layout(); plt.show()

## Module 3 — Full Pipeline Validation & Submission

In [ ]:
# --- Full pipeline: predicted mask -> path, evaluated against GT mask ---
print("Full pipeline validation (train_041 – train_050)\n")
print(f"{'ID':<12} {'SegIoU':>7} {'GT Len':>8} {'PredLen':>8} {'Diff':>7} "
      f"{'ViolPt':>7} {'ViolSeg':>8} {'Time':>6}")
print("-" * 75)

val_results = []
for i in range(40, 50):
    m  = ref_meta[i]
    img_id = m["id"]

    image   = load_image(REF_SATS / f"{img_id}.tiff")
    gt_mask = load_mask(REF_MAPS  / f"{img_id}_map.tiff")
    gt_sol  = load_solution(REF_SOLS / f"{img_id}_solution.json")

    t0 = time.time()

    # 1. predict mask
    pred_mask = postprocess_mask(predict_mask(model, image))

    # 2. segmentation IoU
    inter = (pred_mask & gt_mask).sum()
    union = (pred_mask | gt_mask).sum()
    seg_iou = inter / (union + 1e-8)

    # 3. pathfinding on predicted mask
    cost_map  = build_cost_map(pred_mask)
    pred_path = find_path(cost_map, m["start"], m["goal"], pred_mask)
    dt = time.time() - t0

    # 4. evaluate path against GT mask
    gt_len   = compute_path_length(gt_sol)
    pred_len = compute_path_length(pred_path)
    vp, vs   = count_violations(pred_path, gt_mask)

    val_results.append(dict(id=img_id, iou=seg_iou, gt_len=gt_len,
                            pred_len=pred_len, vp=vp, vs=vs, dt=dt))

    print(f"{img_id:<12} {seg_iou:>7.4f} {gt_len:>8.0f} {pred_len:>8.0f} "
          f"{pred_len-gt_len:>+7.0f} {vp:>7} {vs:>8} {dt:>5.1f}s")

# Summary
avg_iou  = np.mean([r["iou"]  for r in val_results])
avg_vp   = np.mean([r["vp"]   for r in val_results])
avg_vs   = np.mean([r["vs"]   for r in val_results])
avg_diff = np.mean([r["pred_len"] - r["gt_len"] for r in val_results])
print("-" * 75)
print(f"{'AVERAGE':<12} {avg_iou:>7.4f} {'':>8} {'':>8} {avg_diff:>+7.0f} "
      f"{avg_vp:>7.1f} {avg_vs:>8.1f}")

In [ ]:
# --- Generate predicted masks for all test images ---
print("Predicting masks for test images ...\n")

test_predictions = {}
for m in test_meta:
    img_id = m["id"]
    image  = load_image(TEST_SATS / f"{img_id}.tiff")

    t0 = time.time()
    pred = postprocess_mask(predict_mask(model, image))
    dt = time.time() - t0

    test_predictions[img_id] = pred
    print(f"{img_id}: {100*pred.mean():.1f}% road, {dt:.1f}s")

print(f"\nAll {len(test_predictions)} test masks ready.")

In [ ]:
# --- Visualize test mask predictions ---
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
for i, m in enumerate(test_meta):
    r, c = i // 5, i % 5
    image = load_image(TEST_SATS / f"{m['id']}.tiff")
    pred  = test_predictions[m["id"]]

    overlay = image.copy().astype(np.float32)
    overlay[pred] = overlay[pred] * 0.5 + np.array([0, 255, 0], dtype=np.float32) * 0.5
    overlay = overlay.clip(0, 255).astype(np.uint8)

    axes[r, c].imshow(overlay)
    axes[r, c].set_title(m["id"])
    axes[r, c].axis("off")

plt.suptitle("Test predictions (road overlay in green)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# --- Submission generation ---

def generate_submission(test_meta_with_coords, test_predictions,
                        output_path="submission.json"):
    """Generate submission JSON when test start/goal coords are available."""
    submission = []
    for m in test_meta_with_coords:
        img_id = m["id"]
        pred_mask = test_predictions[img_id]
        cost_map  = build_cost_map(pred_mask)
        path      = find_path(cost_map, m["start"], m["goal"], pred_mask)

        submission.append({"id": img_id, "path": path})
        plen = compute_path_length(path)
        vp, vs = count_violations(path, pred_mask)
        print(f"{img_id}: len={plen:.0f}  pts={len(path)}  "
              f"self-violations={vp}pt/{vs}seg")

    with open(output_path, "w") as f:
        json.dump(submission, f)
    print(f"\nSaved {output_path}  ({len(submission)} entries)")
    return submission

# ---- When test coords are available, uncomment: ----
# test_meta_updated = load_metadata(DATA_ROOT / "test_metadata.json")
# submission = generate_submission(test_meta_updated, test_predictions)

print("Submission generator ready.")
print("When test start/goal coords are released, load updated metadata and call generate_submission().")